## 1 Import python libraries and download origin dataset

In [1]:
import os
import pandas as pd
import numpy as np

# Định nghĩa đường dẫn đến dữ liệu gốc và dữ liệu đã xử lý
RAW_DATA_PATH = "../datasets/raw/top-spotify-songs-2023.csv"
PROCESSED_DATA_PATH = "../datasets/processed/cleaned_spotify_2023.csv"

print("--- Downloading origin dataset ---")
# sử dụng encoding 'latin-1' để tránh lỗi khi đọc các ký tự đặc biệt trong tên bài hát hoặc nghệ sĩ.
# latin-1 giúp đọc được các ký tự đặc biệt mà không gây lỗi
df = pd.read_csv(RAW_DATA_PATH, encoding='latin-1')

# Hiển thị thông tin cơ bản về dataset
print(f"Initial dataset size: {df.shape[0]} rows, {df.shape[1]} columns\n")
print(df.info())

--- Downloading origin dataset ---
Initial dataset size: 953 rows, 24 columns

<class 'pandas.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   track_name            953 non-null    str  
 1   artist(s)_name        953 non-null    str  
 2   artist_count          953 non-null    int64
 3   released_year         953 non-null    int64
 4   released_month        953 non-null    int64
 5   released_day          953 non-null    int64
 6   in_spotify_playlists  953 non-null    int64
 7   in_spotify_charts     953 non-null    int64
 8   streams               953 non-null    str  
 9   in_apple_playlists    953 non-null    int64
 10  in_apple_charts       953 non-null    int64
 11  in_deezer_playlists   953 non-null    str  
 12  in_deezer_charts      953 non-null    int64
 13  in_shazam_charts      903 non-null    str  
 14  bpm                   953 non-null    

## 2 Sửa lỗi định dạng và ép kiểu dữ liệu cho cột 'streams'

In [2]:
print("\n--- Đang tiền xử lý cột 'streams' ---")

# Phát hiện các hàng không phải là số (lỗi văn bản do lệch hàng khi cào dữ liệu)
# errors='coerce' sẽ tự động chuyển các hàng lỗi thành NaN (Not a Number)
df['streams'] = pd.to_numeric(df['streams'], errors='coerce')

# Đếm số lượng hàng có kiểu dữ liệu không hợp lệ trong cột 'streams'
null_streams_count = df['streams'].isnull().sum()
print(f"Tìm thấy {null_streams_count} hàng có định dạng không hợp lệ trong cột 'streams'.")

# Loại bỏ các hàng có giá trị NaN trong cột 'streams' vì đây là biến mục tiêu quan trọng nhất
df = df.dropna(subset=['streams'])
# Ép kiểu cột sang số nguyên lớn (int64) để tối ưu bộ nhớ và tính toán
df['streams'] = df['streams'].astype('int64')
# Biến đổi Log (Log transformation): xử lý dữ liệu bị lệch (skewed) cho hồi quy OLS
df['log_streams'] = np.log1p(df['streams'])
print("Đã tạo thành công cột 'log_streams' nhằm đáp ứng các giả định của hồi quy tuyến tính.")


--- Đang tiền xử lý cột 'streams' ---
Tìm thấy 1 hàng có định dạng không hợp lệ trong cột 'streams'.
Đã tạo thành công cột 'log_streams' nhằm đáp ứng các giả định của hồi quy tuyến tính.


## 3 Làm sạch dữ liệu bị lỗi trong các cột từ các nền tảng khác

In [3]:
print("\n--- Đang tiền xử lý cột 'in_deezer_playlists' and 'in_shazam_charts' ---")

# Trong tập dữ liệu này, các cột như 'in_deezer_playlists' hoặc 'in_shazam_charts'
# thường chứa dấu phẩy ngăn cách hàng nghìn (ví dụ: "1,234"), điều này khiến Pandas hiểu lầm chúng là kiểu chuỗi (string).
columns_to_clean = [
    'in_spotify_playlists', 'in_apple_playlists', 'in_apple_charts',
    'in_deezer_playlists', 'in_deezer_charts', 'in_shazam_charts'
]

for col in columns_to_clean:
    if col in df.columns:
        # Loại bỏ dấu phẩy ngăn cách hàng nghìn nếu có
        df[col] = df[col].astype(str).str.replace(',', '')
        # Chuyển đổi sang kiểu số, các trường hợp lỗi sẽ chuyển thành NaN
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # Điền các giá trị NaN bằng 0 (giả định rằng bài hát không nằm trong danh sách phát/bảng xếp hạng đó)
        df[col] = df[col].fillna(0).astype('int64')


--- Đang tiền xử lý cột 'in_deezer_playlists' and 'in_shazam_charts' ---


## 4 Xử lý dữ liệu văn bản và xử lý trùng lặp

In [4]:
# Loại bỏ khoảng trắng thừa ở đầu và cuối các cột 'track_name' và 'artist(s)_name' để đảm bảo tính nhất quán
df['track_name'] = df['track_name'].str.strip()
df['artist(s)_name'] = df['artist(s)_name'].str.strip()

## 5 Xử lý giá trị thiếu cho cột phân loại 'key' (tông nhạc)

In [5]:
if 'key' in df.columns:
    df['key'] = df['key'].fillna('Unknown')

## 6 Kiểm tra và xử lý các giá trị thiếu trong các cột phân tích chính

In [6]:
print("\n--- Đang kiểm tra các đặc trưng âm thanh (Audio Features) ---")

# Các cột đặc trưng âm thanh như 'danceability_%', 'energy_%' phải đầy đủ dữ liệu
audio_features = ['danceability_%', 'energy_%', 'valence_%', 'acousticness_%']

# Kiểm tra giá trị rỗng (null) trong các cột này
for feature in audio_features:
    missing = df[feature].isnull().sum()
    if missing > 0:
        print(f"Cột {feature} đang thiếu {missing} hàng. Đang tiến hành loại bỏ chúng.")
        df = df.dropna(subset=[feature])

print("Trạng thái các giá trị rỗng sau khi xử lý:")
print(df[audio_features + ['streams']].isnull().sum())


--- Đang kiểm tra các đặc trưng âm thanh (Audio Features) ---
Trạng thái các giá trị rỗng sau khi xử lý:
danceability_%    0
energy_%          0
valence_%         0
acousticness_%    0
streams           0
dtype: int64


## 7 Xuất dữ liệu đã làm sạch ra tệp CSV mới để sử dụng sau này

In [7]:
print("\n--- Exporting cleaned data ---")

# Tạo thư mục chứa dữ liệu đã xử lý nếu thư mục đó chưa tồn tại
os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)

# Xuất dữ liệu đã làm sạch ra một tệp CSV mới, index=False để không ghi số thứ tự hàng vào tệp
df.to_csv(PROCESSED_DATA_PATH, index=False, encoding='utf-8')

print(f"Completed! Cleaned dataset size: {df.shape[0]} rows, {df.shape[1]} columns.")
print(f"File saved at: {PROCESSED_DATA_PATH}")


--- Exporting cleaned data ---
Completed! Cleaned dataset size: 952 rows, 25 columns.
File saved at: ../datasets/processed/cleaned_spotify_2023.csv
